<div style="font-family:-apple-system,Segoe UI,sans-serif;background:linear-gradient(108deg,#004F49 0 52%,#EB6658 52%);color:#fff;border-radius:16px;padding:26px 30px;">
<div style="font-size:13px;letter-spacing:2px;text-transform:uppercase;opacity:.9;">International Summer School on Generative AI · Rome 2026</div>
<h1 style="margin:8px 0 4px;font-size:34px;">From Graph to Crew — The World Cup Lab</h1>
<div style="font-size:16px;opacity:.95;">A hands-on tour of agentic AI frameworks · Mohammad Dehghani · AI2Lab</div>
</div>

We build the **same task twice**: a World Cup match-analysis system that produces a scouting report.
Once with **CrewAI** (a *crew* you assemble) and once with **LangGraph** (a *graph* you draw).

> **The goal is not the football.** It is to *feel* the difference between the two orchestration styles by building the identical task in both.

## How to use this notebook

1. Run the cells **top to bottom**.
2. Each participant uses **their own API key**, stored as a **Colab secret** (the key icon in the left sidebar). Never paste a key into a shared notebook or commit it to GitHub.
3. The lab ships with **mock match data** so it runs even without web access. A stretch goal swaps in a real search tool.

**Pick your provider** in Step 2. Default is Anthropic Claude; OpenAI and Google Gemini are one line away.

In [ ]:
# =========================================================
# Step 1 · Install the frameworks (quiet)
# =========================================================
!pip -q install crewai langgraph langchain-anthropic langchain-openai langchain-google-genai

In [ ]:
# =========================================================
# Step 2 · Choose a provider and load YOUR api key (as a secret)
# =========================================================
import os, getpass

PROVIDER = "anthropic"          # "anthropic" | "openai" | "google"

MODELS  = {"anthropic": "claude-haiku-4-5-20251001",
           "openai":    "gpt-4o-mini",
           "google":    "gemini-2.5-flash"}
KEY_ENV = {"anthropic": "ANTHROPIC_API_KEY",
           "openai":    "OPENAI_API_KEY",
           "google":    "GOOGLE_API_KEY"}[PROVIDER]

# Prefer Colab Secrets; fall back to a secure prompt for local use.
try:
    from google.colab import userdata
    os.environ[KEY_ENV] = userdata.get(KEY_ENV)
except Exception:
    if not os.environ.get(KEY_ENV):
        os.environ[KEY_ENV] = getpass.getpass(f"Enter your {KEY_ENV}: ")

MODEL = MODELS[PROVIDER]
print(f"Provider: {PROVIDER}  ·  Model: {MODEL}")

In [ ]:
# =========================================================
# Step 3 · A pretty-print helper for clean output in Colab
# =========================================================
from IPython.display import HTML, display

def pretty_report(title, body, accent="#004F49"):
    """Render a titled, styled card. Pass plain text or light HTML in `body`."""
    display(HTML(f"""
    <div style="font-family:-apple-system,Segoe UI,sans-serif;border:1px solid #e3e0d8;
                border-left:6px solid {accent};border-radius:12px;padding:18px 22px;margin:10px 0;
                background:#fbfaf7;box-shadow:0 2px 10px rgba(20,30,40,.06);">
      <div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:{accent};
                  font-weight:700;">{title}</div>
      <div style="font-size:15px;line-height:1.6;color:#1a1a1a;margin-top:8px;
                  white-space:pre-wrap;">{body}</div>
    </div>"""))

pretty_report("Pretty print ready",
              "Reports in this notebook render in clean cards like this one.")

In [ ]:
# =========================================================
# Step 4 · Data source: a tiny mock so the lab runs offline
# (Swap in a real web-search tool later — see the stretch goal.)
# =========================================================
MATCH_DATA = {
    ("Italy", "Brazil"): {
        "form":    "Italy: W W D L W   ·   Brazil: W W W D W",
        "stats":   "Italy 1.6 xG/game, compact mid-block.  Brazil 2.1 xG/game, high press.",
        "tactics": "Italy 3-5-2, patient build-up.  Brazil 4-2-3-1, full-back overlaps.",
    }
}

def get_match_data(team_a, team_b):
    """Return raw match facts for two teams (mock)."""
    return MATCH_DATA.get((team_a, team_b), {
        "form": "(mock) recent form unavailable",
        "stats": "(mock) stats unavailable",
        "tactics": "(mock) tactical notes unavailable"})

TEAM_A, TEAM_B = "Italy", "Brazil"
print(get_match_data(TEAM_A, TEAM_B))

## Build 1 · CrewAI — *a team you assemble*

We write four **job descriptions** (agents), give each a **task**, and let a **sequential process** run them in order.
If you can describe the team, you can build the crew.

In [ ]:
# =========================================================
# Build 1 · CrewAI — agents, tasks, crew
# =========================================================
from crewai import Agent, Task, Crew, Process, LLM

# CrewAI talks to any provider through one model string, e.g. "anthropic/claude-..."
llm = LLM(model=f"{PROVIDER}/{MODEL}")

data = get_match_data(TEAM_A, TEAM_B)

# 1) Agents = role + goal + backstory
scout   = Agent(role="Scout",           goal="Gather raw match data on both teams",
                backstory="A meticulous football scout.",      llm=llm, verbose=True)
stats   = Agent(role="Stats Analyst",   goal="Interpret the numbers",
                backstory="A data-driven analyst.",            llm=llm, verbose=True)
tactics = Agent(role="Tactics Analyst", goal="Read shape and style",
                backstory="A seasoned tactical expert.",       llm=llm, verbose=True)
writer  = Agent(role="Report Writer",   goal="Assemble a clear scouting report",
                backstory="A concise sports writer.",          llm=llm, verbose=True)

# 2) Tasks = description + expected_output + the agent that owns it
t_scout = Task(description=f"Collect data for {TEAM_A} vs {TEAM_B}. Raw facts:\n{data}",
               expected_output="A bullet list of raw facts.",            agent=scout)
t_stats = Task(description="From the scout's facts, find the statistical edge for each side.",
               expected_output="3 to 4 statistical insights.",          agent=stats)
t_tact  = Task(description="From the scout's facts, analyze tactical matchups and key duels.",
               expected_output="3 to 4 tactical insights.",             agent=tactics)
t_write = Task(description="Write a one-page scouting report with a predicted edge.",
               expected_output="A short markdown scouting report.",     agent=writer)

# 3) Crew = agents + tasks + a process, then kick it off
crew = Crew(agents=[scout, stats, tactics, writer],
            tasks=[t_scout, t_stats, t_tact, t_write],
            process=Process.sequential, verbose=True)

crew_result = crew.kickoff()
pretty_report(f"CrewAI scouting report · {TEAM_A} vs {TEAM_B}", str(crew_result), accent="#D9543F")

## Build 2 · LangGraph — *a state machine you draw*

The same task, but now we draw the **flowchart**: a shared **State**, **nodes** that update it,
and a **conditional edge** that can loop back to the scout if data is missing.
If you can draw the flow, you can build the graph.

In [ ]:
# =========================================================
# Build 2 · LangGraph — state, nodes, edges, a loop
# =========================================================
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# One chat model, chosen from the provider you set in Step 2
if PROVIDER == "anthropic":
    from langchain_anthropic import ChatAnthropic
    chat = ChatAnthropic(model=MODEL, temperature=0)
elif PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    chat = ChatOpenAI(model=MODEL, temperature=0)
else:
    from langchain_google_genai import ChatGoogleGenerativeAI
    chat = ChatGoogleGenerativeAI(model=MODEL, temperature=0)

def ask(prompt):                      # tiny LLM helper
    return chat.invoke(prompt).content

# 1) State = a typed object every node reads and updates
class State(TypedDict):
    raw: str
    stats: str
    tactics: str
    report: str
    loops: int

# 2) Nodes = functions state -> partial state update
def scout_node(s):    return {"raw": str(get_match_data(TEAM_A, TEAM_B)),
                              "loops": s.get("loops", 0) + 1}
def stats_node(s):    return {"stats":   ask(f"Give 3 statistical insights from: {s['raw']}")}
def tactics_node(s):  return {"tactics": ask(f"Give 3 tactical insights from: {s['raw']}")}
def write_node(s):    return {"report":  ask("Write a one-page football scouting report.\n"
                                             f"Stats: {s['stats']}\nTactics: {s['tactics']}")}

# 3) Conditional edge = loop back to scout if data is missing (capped, so it always terminates)
def need_more(s):
    return "scout" if (s["loops"] < 2 and "unavailable" in s["raw"]) else "tactics"

# 4) Wire the graph: nodes, edges, the conditional loop, then compile
g = StateGraph(State)
g.add_node("scout", scout_node)
g.add_node("stats", stats_node)
g.add_node("tactics", tactics_node)
g.add_node("write", write_node)
g.add_edge(START, "scout")
g.add_edge("scout", "stats")
g.add_conditional_edges("stats", need_more, {"scout": "scout", "tactics": "tactics"})
g.add_edge("tactics", "write")
g.add_edge("write", END)
app = g.compile()

graph_out = app.invoke({"loops": 0})
pretty_report(f"LangGraph scouting report · {TEAM_A} vs {TEAM_B}", graph_out["report"], accent="#004F49")

## Same task, two architectures — what did you feel?

| | **CrewAI** (crew) | **LangGraph** (graph) |
|---|---|---|
| You wrote | role + goal + task | nodes + edges + state |
| The loop | hidden in the process | **explicit** conditional edge |
| Best when | quick role decomposition | control, branching, loops |

**Pair, think, share:** which build would you reach for in your own research, and why?

### Stretch goals
1. Swap the mock `get_match_data` for a real web-search tool.
2. Add a 5th role/node (an *injury analyst*) to both builds.
3. In LangGraph, make `need_more` loop when any field is empty, and watch the graph re-scout.

<div style="font-family:-apple-system,Segoe UI,sans-serif;color:#5a6b82;font-size:13px;margin-top:12px;">
Slides and resources: <b>ai2lab.ai</b> · Repo: <b>github.com/mdehghani86/graph-to-crew-lab</b>
</div>